# ViSpatialQA — E6: sửa lỗi FR trả DK, stacking lồng, LP cho nhóm B

Tất cả chấm trên **613 câu Human (train + dev)**, dự đoán ngoài mẫu, cùng các fold story như cv của E4 (Human 0.846).

**Phần 1 (CPU):**
- **FR trả DK sai.** Gold Human FR chỉ có 3/149 câu là DK, nhưng ensemble có 8 lỗi và LPX có 16 lỗi vì trả `[7]`. Hai cách sửa:
  - `fr_unknown=abstain`: LP không chứng minh được quan hệ nào thì abstain;
  - `--human-fr-dk avoid`: ensemble Human không trả DK cho FR nếu có nguồn ủng hộ một quan hệ cụ thể.
- **Stacking lồng (nested).** Việc chọn trọng số hay stacking cho từng loại câu được làm bằng cv bên trong fold, nên con số hybrid không còn lạc quan.
- **Điểm riêng nhóm A và nhóm B.** Nhóm A là câu có `reasoning_type`, chiếm khoảng 90% public test. Nhóm B là câu không có, chiếm 68/94 lỗi.

**Phần 2 (LP cho nhóm B):**
- Chẩn đoán vì sao world không đọc được, và đọc world "khoan dung" (`lenient`: bỏ fact lỗi thay vì bỏ cả world).
- Parser luật hiểu thêm "nằm giữa" và cụm danh từ lồng mệnh đề (đọc được 87% câu Human, nhóm B 76%). Phần này có hiệu lực ngay ở Phần 1, vì LPX dùng `rule_plus`.
- **GPU:** Qwen3-32B chép lại world cho những story có ít hơn 2 world đọc được. Bước này tạo ra nguồn **LPZ**.

### Thời gian ước tính
| Bước | Ước tính |
|---|---|
| Chuẩn bị (clone, dữ liệu, dự đoán E4 từ Drive) | ~1 phút |
| Phần 1 (CPU): chẩn đoán world, grid LP, 3 biến thể × stacking lồng, phân tích lỗi | 8–12 phút |
| Phần 2: cài vLLM + nạp Qwen3-32B | ~6 phút |
| Phần 2: chép lại world cho các story hỏng (8 mẫu mỗi story) | 5–15 phút, tùy số story hỏng in ở mục 3 |
| Phần 2: giải lại + stacking | ~3 phút |
| **Tổng** | **~10 phút chỉ CPU; ~25–40 phút có GPU** |

## 1. CONFIG

In [ ]:
CONFIG = dict(
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/e6-fr-dk-groupb",
    WORK_DIR="/content/VLSP-Task5",
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    BRANCHES="F=F2C,F1 S=S L=LCOT P={P}",             # {P} = nguồn LP của từng biến thể
    BEST_E4_DIR="e4/trdev_LPX/F+S+L+P",                # auto_submission.json lấy từ đây
    C_GRID="0.1,1,10",
    RUN_GPU=True,                                      # False = chỉ chạy Phần 1 + chẩn đoán (runtime CPU là đủ)
    LP=dict(model="Qwen/Qwen3-32B-FP8", fallback="Qwen/Qwen3-32B", redo_below=2, n_world=8, n_form=4,
            max_tokens=12288, max_model_len=20480, max_num_seqs=64, gpu_memory_utilization=0.90),
)

## 2. Chuẩn bị

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import Markdown, display

C = CONFIG
WORK, DRIVE_OUT = Path(C["WORK_DIR"]), Path(C["DRIVE_BACKUP_DIR"])
START = time.time()


def sh(cmd, capture=True):
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd), flush=True)
    if not capture:
        code = subprocess.run(cmd, cwd=WORK).returncode
    else:
        result = subprocess.run(cmd, cwd=WORK, text=True, capture_output=True)
        print(result.stdout[-20000:], result.stderr[-5000:] if result.returncode else "", flush=True)
        code = result.returncode
    if code:
        raise RuntimeError(f"Lệnh lỗi (exit {code})")


if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if (WORK / ".git").exists():
    for args in (["fetch", "--quiet", "origin", C["BRANCH"]], ["checkout", "--quiet", "-B", C["BRANCH"], f"origin/{C['BRANCH']}"]):
        subprocess.run(["git", "-C", str(WORK), *args], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], C["REPO_URL"], str(WORK)], check=True)
os.chdir(WORK)
print("Code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
for name in ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]:
    if not (WORK / "Data" / name).exists():
        shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, WORK / "Data" / name)
if not (WORK / "Data/splits/human_dev.json").exists():
    sh(["python", "make_splits.py", "--from-manifest"])

PRED, OUT = WORK / "outputs/predictions_e6", WORK / "outputs/e6"
OUT.mkdir(parents=True, exist_ok=True)
for folder in ("predictions_e6", "predictions_e4"):             # resume E6 trước, rồi bổ sung từ E4
    if (DRIVE_OUT / folder).exists():
        for path in (DRIVE_OUT / folder).rglob("*.jsonl"):
            target = PRED / path.relative_to(DRIVE_OUT / folder)
            if not target.exists() and ".raw." not in path.name and "F2C_folds" not in path.parts:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
LP_BASE = json.loads((DRIVE_OUT / "e4/lp_choice.json").read_text())["settings"]
auto_sub = DRIVE_OUT / C["BEST_E4_DIR"] / "auto_submission.json"
need = ["F2C/human_train.jsonl", "LCOT/human_train.jsonl", "LPX/human_train.jsonl", "LP/human_train.parses.jsonl"]
assert all((PRED / n).exists() for n in need), "Thiếu dự đoán E4 trên Drive: chạy run_colab_e4.ipynb trước."


def to_drive():
    for folder in ("predictions_e6", "e6"):
        shutil.copytree(WORK / "outputs" / folder, DRIVE_OUT / folder, dirs_exist_ok=True)


def lp_solve(parse_dir, source, settings, splits=("train", "dev", "test")):
    for split in splits:
        split_file = {"train": "Data/splits/human_train.json", "dev": "Data/splits/human_dev.json",
                      "test": "Data/human_public_test.json"}[split]
        sh(["python", "solve_parsed.py", "--parses", PRED / parse_dir / f"human_{split}.parses.jsonl", "--input", split_file,
            *sum((["--set", s] for s in settings), []), "--source", source, "--output", PRED / source / f"human_{split}.jsonl"])


def stack(tag, p_source, fr_dk):
    out = OUT / tag
    sh(["python", "stack_ensemble.py", "--pred-dir", PRED, "--split", "trdev", "--branches", C["BRANCHES"].format(P=p_source),
        "--human-fr-dk", fr_dk, "--c-grid", C["C_GRID"], "--auto-submission", auto_sub, "--output-dir", out])
    return json.loads((out / "results.json").read_text())


VARIANTS = {}
print(f"Chuẩn bị xong sau {(time.time() - START) / 60:.1f} phút. Quy ước LP của E4: {LP_BASE}")

## 3. Chẩn đoán: world của LP (E4) đọc được bao nhiêu, lỗi vì đâu

In [ ]:
for split in ("train", "dev", "test"):
    print(f"===== {split}")
    sh(["python", "solve_parsed.py", "--parses", PRED / f"LP/human_{split}.parses.jsonl",
        "--input", {"train": "Data/splits/human_train.json", "dev": "Data/splits/human_dev.json",
                    "test": "Data/human_public_test.json"}[split], "--world-report", *sum((["--set", s] for s in LP_BASE), [])])

## 4. Phần 1 (CPU)
4a: LPY = quy ước E4 + `fr_unknown=abstain` (luôn bật) + `lenient` nếu grid trên Human train chọn → nguồn **LPY**. Parser luật mới (nằm giữa, cụm lồng) được dùng tự động.
4b: ba biến thể, mỗi biến thể chấm trọng số / stacking / hybrid lồng, kèm điểm nhóm A/B.

In [ ]:
t0 = time.time()
# fr_unknown=abstain luôn bật: grid LP coi abstain là sai nên không bao giờ chọn nó,
# nhưng mục đích là để ensemble dùng nguồn khác thay vì nhận [7]; hiệu quả được đo ở mức ensemble bên dưới.
base_args = sum((["--set", s] for s in LP_BASE + ["fr_unknown=abstain"]), [])
sh(["python", "solve_parsed.py", "--parses", PRED / "LP/human_train.parses.jsonl", "--input", "Data/splits/human_train.json",
    *base_args, "--grid", "--grid-keys", "lenient", "--best-json", OUT / "lpy_best.json"])
LPY_SETTINGS = json.loads((OUT / "lpy_best.json").read_text())["settings"]
print("Quy ước LPY:", LPY_SETTINGS)
lp_solve("LP", "LPY", LPY_SETTINGS)
sh(["python", "crossfit.py", "combine", "--pred-dir", PRED, "--name", "trdev"])

VARIANTS["E4 (LPX, giữ DK)"] = stack("v_e4", "LPX", "keep")
VARIANTS["E4 + tránh DK FR"] = stack("v_e4_avoid", "LPX", "avoid")
VARIANTS["LPY + tránh DK FR"] = stack("v_lpy_avoid", "LPY", "avoid")
to_drive()
print(f"Phần 1 xong sau {(time.time() - t0) / 60:.1f} phút")

In [ ]:
def summary():
    rows = ["| Biến thể | Cách gộp | YN | FR | FB | CO | **Human cv** | Nhóm A | Nhóm B | Final cv |", "|---|---|---|---|---|---|---|---|---|---|"]
    best = None
    for variant, result in VARIANTS.items():
        for method in ("weighted", "hybrid nested"):
            scores, groups = result["results"][method], result["groups"][method]
            human = sum(scores.values()) / 4
            rows.append(f"| {variant} | {'trọng số' if method == 'weighted' else 'hybrid lồng'} | "
                        + " | ".join(f"{scores[t]:.3f}" for t in ("YN", "FR", "FB", "CO"))
                        + f" | **{human:.4f}** | {groups['A']:.3f} | {groups['B']:.3f} | {(human + 0.9873) / 2:.4f} |")
            if best is None or human > best[2]:
                best = (variant, method, human)
    text = "\n".join(rows) + f"\n\n**Tốt nhất:** {best[0]} — {best[1]} (Human cv {best[2]:.4f})."
    display(Markdown(text))
    (OUT / "summary.md").write_text(text + "\n", encoding="utf-8")
    return best


best = summary()

### 4c. Phân tích lỗi của biến thể tốt nhất ở Phần 1

In [ ]:
p_source = "LPY" if best[0].startswith("LPY") else "LPX"
fr_dk = "avoid" if "tránh" in best[0] else "keep"
sh(["python", "analyze_errors.py", "--pred-dir", PRED, "--split", "trdev", "--branches", C["BRANCHES"].format(P=p_source),
    "--human-fr-dk", fr_dk, "--lp-source", p_source, "--output-dir", OUT / "analysis_part1",
    *sum((["--set", s] for s in (LPY_SETTINGS if p_source == "LPY" else LP_BASE)), [])])
text = (OUT / "analysis_part1/report.md").read_text(encoding="utf-8")
display(Markdown(text[:text.index("## 2.")]))
to_drive()

## 5. Phần 2 (GPU): chép lại world cho các story hỏng → nguồn **LPZ**
Parses E4 được sao sang `LP2/` để giữ nguyên bản cũ; chỉ những story có ít hơn `redo_below` world đọc được mới được sinh thêm `n_world` mẫu (gộp với mẫu cũ).

In [ ]:
if C["RUN_GPU"]:
    t0 = time.time()
    if subprocess.run([sys.executable, "-c", "import vllm"], capture_output=True).returncode != 0:
        sh(["pip", "install", "-q", "vllm"], capture=False)
        subprocess.run(["pip", "uninstall", "-y", "-q", "torchaudio", "torchao"])
    os.environ.setdefault("VLLM_USE_DEEP_GEMM", "0"); os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
    for split in ("train", "dev", "test"):
        for suffix in (".jsonl", ".parses.jsonl"):
            source, target = PRED / "LP" / f"human_{split}{suffix}", PRED / "LP2" / f"human_{split}{suffix}"
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, target)
    L = C["LP"]
    jobs = sum((["--job", f"human,{f},{PRED / 'LP2' / f'human_{s}.jsonl'}"] for s, f in
                (("train", "Data/splits/human_train.json"), ("dev", "Data/splits/human_dev.json"), ("test", "Data/human_public_test.json"))), [])
    cmd = ["python", "llm_parse.py", "--source", "LP2", "--redo-worlds-below", L["redo_below"], "--n-world", L["n_world"],
           "--n-form", L["n_form"], "--max-tokens", L["max_tokens"], "--max-model-len", L["max_model_len"],
           "--max-num-seqs", L["max_num_seqs"], "--gpu-memory-utilization", L["gpu_memory_utilization"],
           "--fallback-model", "", "--run-name", "E6-LP2-redo-worlds", *sum((["--set", s] for s in LPY_SETTINGS), []), *jobs]
    try:
        sh(cmd + ["--model-name", L["model"]], capture=False)
    except RuntimeError:
        print(f"⚠️ {L['model']} không chạy được -> thử {L['fallback']}")
        sh(cmd + ["--model-name", L["fallback"]], capture=False)
    for split in ("train", "dev", "test"):
        sh(["python", "solve_parsed.py", "--parses", PRED / f"LP2/human_{split}.parses.jsonl", "--input",
            {"train": "Data/splits/human_train.json", "dev": "Data/splits/human_dev.json", "test": "Data/human_public_test.json"}[split],
            "--world-report", *sum((["--set", s] for s in LPY_SETTINGS), [])])
    lp_solve("LP2", "LPZ", LPY_SETTINGS)
    sh(["python", "crossfit.py", "combine", "--pred-dir", PRED, "--name", "trdev"])
    VARIANTS["LPZ + tránh DK FR"] = stack("v_lpz_avoid", "LPZ", "avoid")
    to_drive()
    print(f"Phần 2 xong sau {(time.time() - t0) / 60:.1f} phút")
best = summary()

## 6. File nộp của biến thể tốt nhất
- Nếu cách gộp tốt nhất là **trọng số**: `compare_branches.py` tune trên 613 câu và ghi file nộp.
- Nếu là **hybrid lồng**: dùng file `hybrid/` mà `stack_ensemble.py` đã ghi. Ở file nộp, mỗi loại câu chọn cách gộp theo cv trên toàn bộ 613 câu.

In [ ]:
variant, method, human = best
p_source = {"E4 (LPX, giữ DK)": "LPX", "E4 + tránh DK FR": "LPX", "LPY + tránh DK FR": "LPY", "LPZ + tránh DK FR": "LPZ"}[variant]
fr_dk = "avoid" if "tránh" in variant else "keep"
tag = {"E4 (LPX, giữ DK)": "v_e4", "E4 + tránh DK FR": "v_e4_avoid", "LPY + tránh DK FR": "v_lpy_avoid", "LPZ + tránh DK FR": "v_lpz_avoid"}[variant]
final_dir = OUT / "submission"
if method == "weighted":
    sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", OUT / "compare_best", "--branches",
        C["BRANCHES"].format(P=p_source), "--tune-splits", "human=trdev", "--human-fr-dk", fr_dk, "--no-use-indifinite"])
    shutil.copytree(OUT / "compare_best/F+S+L+P", final_dir, dirs_exist_ok=True)
else:
    shutil.copytree(OUT / tag / "hybrid", final_dir, dirs_exist_ok=True)
sh(["python", "validate_submission.py", final_dir / "human_submission.json", "--reference", "Data/human_public_test.json"])
sh(["python", "validate_submission.py", final_dir / "auto_submission.json", "--reference", "Data/auto_public_test.json"])
to_drive()
print(f"Bài nộp: {variant} — {method} (Human cv {human:.4f}) -> Drive outputs/e6/submission/. "
      f"Tổng {(time.time() - START) / 60:.1f} phút")